# UHI Modelling V2

I am attempting a revised workflow for modelling UHI.
First, the model is now going to include weather variables obtained from OpenMeteo. The data will then be aggregated with the Landsat data and used to build the model with the workflow below:  
1. Identify the world's climates from an authority (like the Trewartha climate classification).
2. Get substantial satellite (which will eventually include those those climate categories) and weather data on countries in those regions across the months (seasons) in a particular year.
3. Train a model on the data, implementing train-validation-test split, and predict UHI intensity on the test set.
4. Group the test examples by climate and compute the RMSE scores across the climates.
5. Cluster the UHI features to identify the types and categories of UHI.
6. Then I, human, assess the errors and prediction accuracies across each manufactured UHI cluster and how they vary across climates.

***"This project predicts urban heat island intensity using satellite and environmental data and evaluates how prediction reliability and errors vary across global climates and urban thermal types."***

## Testing aggregation with minimal data

To test the flow of data:  
1. From the various countries
2. In the various climates
3. Once a week from Jan 1 2025 to Dec 31 2025
4. From Earth Engine (Landsat) and OpenMeteo

I will be using as minimal data as possible. This will look like:  
1. Lagos, Ontario, Helsinki, and Tehran
2. Aw, Dc, Dcb, Bsk
3. Once a month Jan 1 2025 to Dec 31 2025
4. From Earth Engine and OpenMeteo

## Fetching from Open-Meteo

Again, the cities I want to sample are:
1. Lagos, Nigeria
2. Ontario, Canada
3. Tehran, Iran
4. Helsinki, Finland

The variables I am getting from Open-meteo are:
1. Cloud Cover (Low)
2. Air Temperature (2m)
3. Ralative Humidity
4. Precipitation
5. Wind Speed (10m)

In [1]:
import ee
import geopandas as gpd

ee.Authenticate()
ee.Initialize()

In [2]:
from spectral import get_spectral, get_viirs

cities = [
    {"code": "CAN", "name": "Ontario"},
    {"code": "IRN", "name": "Tehran"},
    {"code": "NGA", "name": "Lagos"},
	{"code": "FIN", "name": "Uusimaa"},
]

date_range = ("2025-01-01", "2025-12-31")

for city in cities:
    get_spectral(city['code'], city['name'], date_range)
    get_viirs(city['code'], date_range)


c:\Software Projects\Data Projects\uhi-modelling\uhenv\Lib\site-packages\geemap\conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Sample points already exist at data/CAN_sample_points.csv.

VIIRS data already exists at data/CAN_viirs_features.csv.

Sample points already exist at data/IRN_sample_points.csv.

VIIRS data already exists at data/IRN_viirs_features.csv.

Sample points already exist at data/NGA_sample_points.csv.

VIIRS data already exists at data/NGA_viirs_features.csv.

Sample points already exist at data/FIN_sample_points.csv.

VIIRS data already exists at data/FIN_viirs_features.csv.



In [3]:
import pandas as pd

for city in cities:
    df = pd.read_csv(f"data/{city['code']}_sample_points.csv")
    urban = (df["LandCover"] == 50).sum()
    rural = (df["LandCover"] != 50).sum()
    print(f"{city['name']}: {urban} urban, {rural} rural, total: {len(df)}")

Ontario: 100 urban, 100 rural, total: 200
Tehran: 100 urban, 100 rural, total: 200
Lagos: 100 urban, 100 rural, total: 200
Uusimaa: 100 urban, 100 rural, total: 200


In [4]:
for city in cities:
    df = pd.read_csv(f"data/{city['code']}_viirs_features.csv")
    print(f"{city['name']}: {len(df)} rows")
    print(df[["LST_day", "LST_night", "NDVI", "NDBI", "MNDWI", "SAVI", "Albedo"]].isnull().sum())

Ontario: 2400 rows
LST_day      0
LST_night    0
NDVI         0
NDBI         0
MNDWI        0
SAVI         0
Albedo       0
dtype: int64
Tehran: 2400 rows
LST_day      0
LST_night    0
NDVI         0
NDBI         0
MNDWI        0
SAVI         0
Albedo       0
dtype: int64
Lagos: 2229 rows
LST_day      0
LST_night    0
NDVI         0
NDBI         0
MNDWI        0
SAVI         0
Albedo       0
dtype: int64
Uusimaa: 2198 rows
LST_day      0
LST_night    0
NDVI         0
NDBI         0
MNDWI        0
SAVI         0
Albedo       0
dtype: int64


In [5]:
for city in cities:
    df = pd.read_csv(f"data/{city['code']}_viirs_features.csv")
    missing_months = set(range(1, 13)) - set(df["month"].unique())
    print(f"{city['name']}: missing months {missing_months}")

Ontario: missing months set()
Tehran: missing months set()
Lagos: missing months set()
Uusimaa: missing months {6}


In [6]:
from weather import process_city_weather

for city in cities:
    process_city_weather(city['code'], date_range)

Fetching weather for Point 1/200 in CAN...
Successfully fetched data for 43.43184186329071, -80.48249509241631, 2025-01-01-2025-12-31
Fetching weather for Point 2/200 in CAN...
Successfully fetched data for 43.25267352140154, -79.83325284270528, 2025-01-01-2025-12-31
Fetching weather for Point 3/200 in CAN...
Successfully fetched data for 43.588787053240345, -79.63683468234007, 2025-01-01-2025-12-31
Fetching weather for Point 4/200 in CAN...
Successfully fetched data for 43.08692016475747, -79.13942172062603, 2025-01-01-2025-12-31
Fetching weather for Point 5/200 in CAN...
Successfully fetched data for 43.25804596763376, -79.86192101445111, 2025-01-01-2025-12-31
Fetching weather for Point 6/200 in CAN...
Successfully fetched data for 43.14484116517082, -79.19054711138577, 2025-01-01-2025-12-31
Fetching weather for Point 7/200 in CAN...
Successfully fetched data for 43.34633839140493, -79.79461655719854, 2025-01-01-2025-12-31
Fetching weather for Point 8/200 in CAN...
Successfully fetch

In [7]:
for city in cities:
    df = pd.read_csv(f"data/{city['code']}_Full_UHI_Data.csv")
    print(f"{city['name']}: {len(df)} rows")

Ontario: 4800 rows
Tehran: 4800 rows
Lagos: 4800 rows
Uusimaa: 4800 rows


Now that we have fetched our data, we will add the `city` column and concatenate all four datasets.

In [10]:
# Step 1 — Consolidate all cities
viirs_dfs, weather_dfs = [], []

for city in cities:
    viirs_df = pd.read_csv(f"data/{city['code']}_viirs_features.csv")
    viirs_df["city"] = city["name"]
    viirs_dfs.append(viirs_df)

    weather_df = pd.read_csv(f"data/{city['code']}_Full_UHI_Data.csv")
    weather_df["city"] = city["name"]
    weather_dfs.append(weather_df)

viirs_data = pd.concat(viirs_dfs, ignore_index=True)
weather_data = pd.concat(weather_dfs, ignore_index=True)

# Step 2 — Parse date and extract month and time
weather_data["date"] = pd.to_datetime(weather_data["date"], utc=True)
weather_data["month"] = weather_data["date"].dt.month
weather_data["time"] = weather_data["date"].dt.hour.map({1: "night", 13: "day"})


# Step 4 — Merge VIIRS into uhi_data on lat/lon + city + month
uhi_data = weather_data.merge(
    viirs_data,
    on=["latitude", "longitude", "city", "month"],
    how="left"
)

print(uhi_data.shape)
print(uhi_data[["LandCover", "Elevation", "LST_day", "LST_night", "NDVI", "NDBI", "MNDWI", "SAVI", "Albedo"]].isnull().sum())

(19200, 20)
LandCover      0
Elevation      0
LST_day      746
LST_night    746
NDVI         746
NDBI         746
MNDWI        746
SAVI         746
Albedo       746
dtype: int64


In [11]:
print(uhi_data[uhi_data["LST_day"].isnull()]["city"].value_counts())
print(uhi_data[uhi_data["LST_day"].isnull()]["month"].value_counts())

city
Uusimaa    404
Lagos      342
Name: count, dtype: int64
month
6     426
3      48
8      36
7      30
12     28
4      28
9      28
2      26
10     26
5      26
11     26
1      18
Name: count, dtype: int64


In [12]:
uhi_data = uhi_data.sort_values(["city", "latitude", "longitude", "month", "time"]).reset_index(drop=True)

for col in ["LST_day", "LST_night", "NDVI", "NDBI", "MNDWI", "SAVI", "Albedo"]:
    uhi_data[col] = uhi_data.groupby(["city", "latitude", "longitude"])[col].transform(
        lambda x: x.interpolate(method="linear", limit_direction="both")
    )

print(uhi_data[["LST_day", "LST_night", "NDVI", "NDBI", "MNDWI", "SAVI", "Albedo"]].isnull().sum())

LST_day      192
LST_night    192
NDVI         192
NDBI         192
MNDWI        192
SAVI         192
Albedo       192
dtype: int64


In [13]:
null_mask = uhi_data["LST_day"].isnull()
print(uhi_data[null_mask]["city"].value_counts())
print(uhi_data[null_mask].groupby(["city", "latitude", "longitude"])["month"].apply(list))

city
Lagos    192
Name: count, dtype: int64
city   latitude  longitude
Lagos  6.478715  3.496877     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.479197  3.438773     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.497516  3.556839     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.503954  3.539145     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.512168  3.415794     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.515417  3.518704     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.517941  3.572257     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
       6.577391  3.676986     [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, ...
Name: month, dtype: object


In [14]:
null_pixels = uhi_data[uhi_data["LST_day"].isnull()][["city", "latitude", "longitude"]].drop_duplicates()

uhi_data = uhi_data.merge(
    null_pixels.assign(drop=True),
    on=["city", "latitude", "longitude"],
    how="left"
)
uhi_data = uhi_data[uhi_data["drop"].isnull()].drop(columns=["drop"])

print(uhi_data.shape)
print(uhi_data[["LST_day", "LST_night"]].isnull().sum())
print(uhi_data["city"].value_counts())

(19008, 20)
LST_day      0
LST_night    0
dtype: int64
city
Ontario    4800
Tehran     4800
Uusimaa    4800
Lagos      4608
Name: count, dtype: int64


In [15]:
lagos = uhi_data[uhi_data["city"] == "Lagos"]
print(lagos[["latitude", "longitude", "LandCover"]].drop_duplicates()["LandCover"].value_counts())
print(f"Urban: {(lagos['LandCover'] == 50).sum() // 24}")
print(f"Rural: {(lagos['LandCover'] != 50).sum() // 24}")

LandCover
50.0    100
10.0     40
80.0     23
30.0     14
90.0      7
20.0      5
40.0      1
60.0      1
95.0      1
Name: count, dtype: int64
Urban: 100
Rural: 92


## Rest of the workflow.

In [16]:
uhi_data["is_urban"] = (uhi_data["LandCover"] == 50).astype(int)

urban = uhi_data[uhi_data["is_urban"] == 1].groupby(["city", "month", "time"]).agg(
    LST_day_urban=("LST_day", "mean"),
    LST_night_urban=("LST_night", "mean"),
    airtemp_urban=("air_temperature", "mean"),
    n_urban=("LST_day", "count")
).reset_index()

rural = uhi_data[uhi_data["is_urban"] == 0].groupby(["city", "month", "time"]).agg(
    LST_day_rural=("LST_day", "mean"),
    LST_night_rural=("LST_night", "mean"),
    airtemp_rural=("air_temperature", "mean"),
    n_rural=("LST_day", "count")
).reset_index()

uhi_intensity = urban.merge(rural, on=["city", "month", "time"], how="inner")

uhi_intensity["SUHI_day"] = uhi_intensity["LST_day_urban"] - uhi_intensity["LST_day_rural"]
uhi_intensity["SUHI_night"] = uhi_intensity["LST_night_urban"] - uhi_intensity["LST_night_rural"]
uhi_intensity["AUHI"] = uhi_intensity["airtemp_urban"] - uhi_intensity["airtemp_rural"]

uhi_intensity = uhi_intensity.drop(columns=[
    "LST_day_urban", "LST_day_rural",
    "LST_night_urban", "LST_night_rural",
    "airtemp_urban", "airtemp_rural"
])

print(uhi_intensity.shape)
print(uhi_intensity[["SUHI_day", "SUHI_night", "AUHI"]].describe())
print(uhi_intensity[["n_urban", "n_rural"]].value_counts())

(96, 8)
        SUHI_day  SUHI_night       AUHI
count  96.000000   96.000000  96.000000
mean    0.064075    0.040410   1.245613
std     0.072681    0.037440   1.958347
min    -0.058016   -0.008752  -3.187605
25%     0.014779    0.006326  -0.089142
50%     0.062932    0.027761   0.381726
75%     0.120517    0.081922   3.198875
max     0.210861    0.093826   7.611895
n_urban  n_rural
100      100        72
         92         24
Name: count, dtype: int64


Let's clean up the data:
1. Convert date to datetime
2. Convert city to category
3. Convert LandCover to category

In [7]:
uhi_data["date"] = pd.to_datetime(uhi_data["date"], utc = True)

uhi_data.loc[uhi_data["date"].dt.hour.isin(range(6, 11)), "time"] = "morning"
uhi_data.loc[uhi_data["date"].dt.hour.isin(range(12, 15)), "time"] = "afternoon"
uhi_data.loc[uhi_data["date"].dt.hour.isin(range(16, 23)), "time"] = "evening"
    
print(uhi_data["time"].value_counts())
print(uhi_data["time"].isnull().sum())

time
morning      9600
afternoon    9600
evening      9600
Name: count, dtype: int64
0


I am going to experiment with clustering the points to identify the inherent climate classifications they belong to, because I cannot find a fine-grained Trewartha classification.

In [ ]:
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans

# pixel_profiles = uhi_data.groupby(["latitude", "longitude"]).agg(
#     mean_temp=("air_temperature", "mean"),
#     mean_humidity=("humidity", "mean"),
#     total_precip=("precipitation", "sum"),
#     mean_wind=("wind_speed", "mean"),
#     mean_cloud=("cloud_cover_low", "mean")
# ).reset_index()

# features = ["mean_temp", "mean_humidity", "total_precip", "mean_wind", "mean_cloud"]

# scaler = StandardScaler()
# scaled = scaler.fit_transform(pixel_profiles[features])


# inertias = []
# k_range = range(2, 11)
# for k in k_range:
#     km = KMeans(n_clusters=k, random_state=42)
#     km.fit(scaled)
#     inertias.append(km.inertia_)

# import matplotlib.pyplot as plt
# plt.plot(k_range, inertias, marker="o")
# plt.xlabel("Number of clusters")
# plt.ylabel("Inertia")
# plt.title("Elbow Method")
# plt.show()

# print(intertias)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pixel_profiles = uhi_data.groupby(["latitude", "longitude"]).agg(
    mean_temp=("air_temperature", "mean"),
    mean_humidity=("humidity", "mean"),
    total_precip=("precipitation", "sum"),
    mean_wind=("wind_speed", "mean"),
    mean_cloud=("cloud_cover_low", "mean")
).reset_index()

features = ["mean_temp", "mean_humidity", "total_precip", "mean_wind", "mean_cloud"]
scaler = StandardScaler()
scaled = scaler.fit_transform(pixel_profiles[features])

km = KMeans(n_clusters=4, random_state=42)
pixel_profiles["climate"] = km.fit_predict(scaled)

uhi_data = uhi_data.merge(
    pixel_profiles[["latitude", "longitude", "climate"]],
    on=["latitude", "longitude"],
    how="left"
)

print(uhi_data["climate"].value_counts())
print(uhi_data["climate"].isnull().sum())

climate
1    11556
0     7200
2     7200
3     2844
Name: count, dtype: int64
0


Now, deriving SUHI and AUHI.

In [12]:
uhi_data["is_urban"] = (uhi_data["LandCover"] == 50).astype(int)

# Separate urban and rural
urban = uhi_data[uhi_data["is_urban"] == 1].groupby(["city", "date", "time"]).agg(
    LST_urban=("LST", "mean"),
    airtemp_urban=("air_temperature", "mean"),
    n_urban=("LST", "count")
).reset_index()

rural = uhi_data[uhi_data["is_urban"] == 0].groupby(["city", "date", "time"]).agg(
    LST_rural=("LST", "mean"),
    airtemp_rural=("air_temperature", "mean"),
    n_rural=("LST", "count")
).reset_index()

# Merge and compute UHI intensity
uhi_intensity = urban.merge(rural, on=["city", "date", "time"], how="inner")
uhi_intensity["SUHI"] = uhi_intensity["LST_urban"] - uhi_intensity["LST_rural"]
uhi_intensity["AUHI"] = uhi_intensity["airtemp_urban"] - uhi_intensity["airtemp_rural"]

print(uhi_intensity.shape)
print(uhi_intensity[["SUHI", "AUHI"]].describe())
print(uhi_intensity[["n_urban", "n_rural"]].value_counts())

(144, 11)
             SUHI        AUHI
count  144.000000  144.000000
mean     1.055471    1.150687
std      1.344073    2.010427
min     -0.562931   -6.276605
25%     -0.015930   -0.072750
50%      0.958839    0.358375
75%      2.030240    2.989625
max      2.867138    7.194395
n_urban  n_rural
100      100        144
Name: count, dtype: int64


In [18]:
print(uhi_intensity[uhi_intensity["city"] == "Ontario"].head())

       city                      date       time   LST_urban  airtemp_urban  \
36  Ontario 2025-01-01 08:00:00+00:00    morning  284.912131       1.648435   
37  Ontario 2025-01-01 14:00:00+00:00  afternoon  284.912131       1.512435   
38  Ontario 2025-01-01 20:00:00+00:00    evening  284.912131       2.072435   
39  Ontario 2025-02-01 08:00:00+00:00    morning  284.912131     -11.629565   
40  Ontario 2025-02-01 14:00:00+00:00  afternoon  284.912131     -14.187565   

    n_urban   LST_rural  airtemp_rural  n_rural      SUHI      AUHI  
36      100  282.044993       -0.55296      100  2.867138  2.201395  
37      100  282.044993        0.85654      100  2.867138  0.655895  
38      100  282.044993        1.09154      100  2.867138  0.980895  
39      100  282.044993      -12.58896      100  2.867138  0.959395  
40      100  282.044993      -14.59346      100  2.867138  0.405895  


In [18]:
uhi_data[uhi_data["is_urban"] == 1].groupby("city").size()

city
Lagos      1620
Ontario      36
Tehran      396
Uusimaa     432
dtype: int64